In [1]:
# ⚠️  LightOnOCR-2 requires transformers >= 5.0 (latest).
#     This conflicts with Surya OCR which needs transformers < 5.0.
#     Run this notebook in a fresh kernel / separate venv from Surya OCR.ipynb.
!uv pip install "transformers>=5.0.0" torch pillow pandas tqdm -q

# LightOnOCR-2-1B Evaluation on Synthetic Handwritten Data

This notebook evaluates **LightOnOCR-2-1B** performance on synthetic handwritten German nursing notes.

LightOnOCR-2 is an end-to-end 1B-parameter vision-language model — it takes a document image and outputs clean, naturally ordered text in a single pass (no separate detection + recognition steps).

**Metrics Calculated:**
- Accuracy (sequence similarity)
- CER (Character Error Rate)
- WER (Word Error Rate)
- NED (Normalized Edit Distance)
- Vocabulary Precision / Recall / F1 (unique word overlap)

**Output:**
- Individual results: `results/metrics/lightonocr/individual/`
- Summary CSV: `results/metrics/lightonocr/summary.csv`
- Summary JSON: `results/metrics/lightonocr/summary.json`
- Detailed results JSON: `results/metrics/lightonocr/detailed_results.json`

## 1. Setup and Imports

In [2]:
import sys
import json
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.evaluation.metrics import calculate_all_metrics, format_metrics_display

print("✓ All imports successful")
print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

/home/bashar/bachelor-thesis-nursing-agents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
Project root: /home/bashar/bachelor-thesis-nursing-agents
PyTorch version: 2.10.0+cu128


## 2. Configuration

In [3]:
# Paths
IMAGES_DIR = project_root / "data" / "synthetic" / "output" / "images"
LABELS_DIR = project_root / "data" / "synthetic" / "output" / "labels"
RESULTS_DIR = project_root / "results" / "metrics" / "lightonocr"
INDIVIDUAL_RESULTS_DIR = RESULTS_DIR / "individual"

# Create output directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INDIVIDUAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==================================================================================
# EVALUATION STRATEGY - Choose one:
# ==================================================================================
# Option A: Use fixed test set (RECOMMENDED for fair comparison)
USE_TEST_SET = True
TEST_SET_FILE = project_root / "data" / "test_set_600.json"  # or test_set_500.json

# Option B: Random sampling (NOT recommended for comparing OCR methods)
# USE_TEST_SET = False
# LIMIT_SAMPLES = None  # Set to a number (e.g., 10) for testing, None for all images
# ==================================================================================

# Model config
MODEL_ID = "lightonai/LightOnOCR-2-1B"
MAX_NEW_TOKENS = 2048

print(f"Images directory: {IMAGES_DIR}")
print(f"Labels directory: {LABELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

if USE_TEST_SET:
    if TEST_SET_FILE.exists():
        print(f"✓ Using FIXED TEST SET: {TEST_SET_FILE}")
        print("  (Ensures fair comparison across OCR methods)")
    else:
        print(f"⚠ TEST SET FILE NOT FOUND: {TEST_SET_FILE}")
        print("  Run: python scripts/create_test_set.py --strategy balanced-600 --output data/test_set_600.json")
        USE_TEST_SET = False
        LIMIT_SAMPLES = None
else:
    print(f"⚠ Using random sampling: {LIMIT_SAMPLES if 'LIMIT_SAMPLES' in globals() and LIMIT_SAMPLES else 'All images'}")
    print("  Note: This may not be fair for comparing different OCR methods!")

Images directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/images
Labels directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/labels
Results directory: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/lightonocr
✓ Using FIXED TEST SET: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
  (Ensures fair comparison across OCR methods)


## 3. Initialize LightOnOCR-2

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float32 if device == "mps" else torch.bfloat16

print(f"Device: {device}  |  dtype: {dtype}")
print(f"Loading model: {MODEL_ID}")
print("This downloads ~2 GB on first run and loads the model into GPU/CPU memory...")

model     = LightOnOcrForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
model.eval()

print("✓ LightOnOCR-2-1B model and processor initialized successfully")

Device: cuda  |  dtype: torch.bfloat16
Loading model: lightonai/LightOnOCR-2-1B
This downloads ~2 GB on first run and loads the model into GPU/CPU memory...


You are using a model of type `mistral3` to instantiate a model of type `lighton_ocr`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|██████████| 532/532 [00:00<00:00, 1489.70it/s]


✓ LightOnOCR-2-1B model and processor initialized successfully


## 4. Load Image and Label Pairs

In [5]:
# Load image-label pairs based on evaluation strategy
valid_pairs = []

if USE_TEST_SET and TEST_SET_FILE.exists():
    # Load from fixed test set
    print(f"Loading test set from: {TEST_SET_FILE}")
    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        test_set_data = json.load(f)

    print(f"Test set contains {test_set_data['test_set_size']} samples")

    for sample in test_set_data['samples']:
        img_path   = project_root / sample['image_path']
        label_path = project_root / sample['label_path']

        if img_path.exists() and label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Missing files for {sample['sample_id']}")

    print(f"✓ Loaded {len(valid_pairs)} samples from FIXED TEST SET")
    print( "  (Same samples will be used for all OCR methods)")

else:
    # Fall back to all images or limited sampling
    print("Loading all images from directory...")
    image_files = sorted(list(IMAGES_DIR.glob("*.png")))

    if 'LIMIT_SAMPLES' in globals() and LIMIT_SAMPLES:
        image_files = image_files[:LIMIT_SAMPLES]

    print(f"Found {len(image_files)} images")

    for img_path in image_files:
        label_path = LABELS_DIR / f"{img_path.stem}.json"
        if label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Warning: No label file found for {img_path.name}")

    print(f"✓ {len(valid_pairs)} valid image-label pairs ready")
    if not USE_TEST_SET:
        print("⚠ WARNING: Not using fixed test set — may not be fair for OCR comparison!")

Loading test set from: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
Test set contains 559 samples
✓ Loaded 559 samples from FIXED TEST SET
  (Same samples will be used for all OCR methods)


## 5. Batch Processing with OCR and Evaluation

In [6]:
results_list    = []
failed_samples  = []

print(f"\n{'=' * 80}")
print("STARTING BATCH OCR PROCESSING  (LightOnOCR-2-1B)")
print(f"{'=' * 80}\n")

for img_path, label_path in tqdm(valid_pairs, desc="Processing images"):
    sample_id = img_path.stem

    try:
        # Load ground truth
        with open(label_path, 'r', encoding='utf-8') as f:
            label_data = json.load(f)

        expected_text = label_data.get('full_text', '')

        # ── Run LightOnOCR-2 ──────────────────────────────────────────────────
        # LightOnOCR uses a chat-template style interface:
        # pass the image path as a URL and let the model generate the text.
        conversation = [{"role": "user", "content": [{"type": "image", "url": str(img_path)}]}]

        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

        generated_ids  = output_ids[0, inputs["input_ids"].shape[1]:]
        predicted_text = processor.decode(generated_ids, skip_special_tokens=True)
        # ─────────────────────────────────────────────────────────────────────

        # Calculate all metrics
        metrics = calculate_all_metrics(expected_text, predicted_text)

        result = {
            'sample_id':        sample_id,
            'image_path':       str(img_path.relative_to(project_root)),
            'expected_text':    expected_text,
            'predicted_text':   predicted_text,
            'expected_length':  len(expected_text),
            'predicted_length': len(predicted_text),
            **metrics,
        }

        results_list.append(result)

        # Save individual result
        individual_result_path = INDIVIDUAL_RESULTS_DIR / f"{sample_id}_result.json"
        with open(individual_result_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"✗ Error processing {sample_id}: {str(e)}")
        failed_samples.append({'sample_id': sample_id, 'error': str(e)})

print(f"\n{'=' * 80}")
print(f"✓ Successfully processed: {len(results_list)} samples")
print(f"✗ Failed: {len(failed_samples)} samples")
print(f"{'=' * 80}")


STARTING BATCH OCR PROCESSING  (LightOnOCR-2-1B)



Processing images: 100%|██████████| 559/559 [1:47:34<00:00, 11.55s/it]  


✓ Successfully processed: 559 samples
✗ Failed: 0 samples


## 6. Create Summary Statistics

In [7]:
if len(results_list) > 0:
    # Create DataFrame for analysis
    df_results = pd.DataFrame([
        {
            'sample_id':        r['sample_id'],
            'accuracy':         r['accuracy'],
            'cer':              r['cer'],
            'wer':              r['wer'],
            'ned':              r['ned'],
            'vocab_precision':  r['vocab_precision'],
            'vocab_recall':     r['vocab_recall'],
            'vocab_f1':         r['vocab_f1'],
            'expected_length':  r['expected_length'],
            'predicted_length': r['predicted_length'],
        }
        for r in results_list
    ])

    def _stat(series):
        return {
            'mean':   float(series.mean()),
            'std':    float(series.std()),
            'min':    float(series.min()),
            'max':    float(series.max()),
            'median': float(series.median()),
        }

    summary_stats = {
        'method':       'LightOnOCR-2-1B',
        'model_id':     MODEL_ID,
        'num_samples':  len(results_list),
        'timestamp':    datetime.now().isoformat(),
        'metrics': {
            'accuracy':       _stat(df_results['accuracy']),
            'cer':            _stat(df_results['cer']),
            'wer':            _stat(df_results['wer']),
            'ned':            _stat(df_results['ned']),
            'vocab_precision': _stat(df_results['vocab_precision']),
            'vocab_recall':   _stat(df_results['vocab_recall']),
            'vocab_f1':       _stat(df_results['vocab_f1']),
        }
    }

    print("Summary Statistics:")
    print("=" * 80)
    for metric_name, stats in summary_stats['metrics'].items():
        print(f"\n{metric_name.upper()}:")
        print(f"  Mean:   {stats['mean']:.2f}%")
        print(f"  Median: {stats['median']:.2f}%")
        print(f"  Std:    {stats['std']:.2f}%")
        print(f"  Range:  [{stats['min']:.2f}%, {stats['max']:.2f}%]")
    print("=" * 80)
else:
    print("⚠ No results to summarize")

Summary Statistics:

ACCURACY:
  Mean:   65.62%
  Median: 82.43%
  Std:    35.15%
  Range:  [0.00%, 100.00%]

CER:
  Mean:   40.21%
  Median: 4.44%
  Std:    139.11%
  Range:  [0.00%, 1939.80%]

WER:
  Mean:   67.14%
  Median: 21.31%
  Std:    203.14%
  Range:  [0.00%, 2106.32%]

NED:
  Mean:   17.16%
  Median: 4.44%
  Std:    23.84%
  Range:  [0.00%, 96.84%]

VOCAB_PRECISION:
  Mean:   69.32%
  Median: 80.00%
  Std:    29.13%
  Range:  [0.00%, 100.00%]

VOCAB_RECALL:
  Mean:   71.57%
  Median: 80.85%
  Std:    27.84%
  Range:  [0.00%, 100.00%]

VOCAB_F1:
  Mean:   70.10%
  Median: 80.00%
  Std:    28.43%
  Range:  [0.00%, 100.00%]


## 7. Save Results

In [8]:
if len(results_list) > 0:
    # Save CSV (for easy analysis in Excel/Python)
    csv_path = RESULTS_DIR / "summary.csv"
    df_results.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✓ Saved CSV summary: {csv_path}")

    # Save summary JSON
    summary_json_path = RESULTS_DIR / "summary.json"
    with open(summary_json_path, 'w', encoding='utf-8') as f:
        json.dump(summary_stats, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved JSON summary: {summary_json_path}")

    # Save detailed results (all data)
    detailed_json_path = RESULTS_DIR / "detailed_results.json"
    with open(detailed_json_path, 'w', encoding='utf-8') as f:
        json.dump(results_list, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved detailed results: {detailed_json_path}")

    # Save failed samples if any
    if failed_samples:
        failed_path = RESULTS_DIR / "failed_samples.json"
        with open(failed_path, 'w', encoding='utf-8') as f:
            json.dump(failed_samples, f, ensure_ascii=False, indent=2)
        print(f"⚠ Saved failed samples log: {failed_path}")

    print(f"\n{'=' * 80}")
    print("ALL RESULTS SAVED SUCCESSFULLY")
    print(f"{'=' * 80}")
    print(f"\nResults location: {RESULTS_DIR}")
    print(f"  - summary.csv ({len(df_results)} rows)")
    print(f"  - summary.json (aggregate statistics)")
    print(f"  - detailed_results.json (full OCR output)")
    print(f"  - individual/ ({len(results_list)} files)")
else:
    print("⚠ No results to save")

✓ Saved CSV summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/lightonocr/summary.csv
✓ Saved JSON summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/lightonocr/summary.json
✓ Saved detailed results: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/lightonocr/detailed_results.json

ALL RESULTS SAVED SUCCESSFULLY

Results location: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/lightonocr
  - summary.csv (559 rows)
  - summary.json (aggregate statistics)
  - detailed_results.json (full OCR output)
  - individual/ (559 files)


## 8. Quick Analysis and Visualization

In [9]:
if len(results_list) > 0:
    # Display top 5 best and worst performing samples
    print("\n" + "=" * 80)
    print("TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    top_5 = df_results.nlargest(5, 'vocab_f1')
    print(top_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))

    print("\n" + "=" * 80)
    print("TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    bottom_5 = df_results.nsmallest(5, 'vocab_f1')
    print(bottom_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))

    # Basic statistics
    print("\n" + "=" * 80)
    print("BASIC STATISTICS")
    print("=" * 80)
    print(f"Samples with accuracy > 90%:       {(df_results['accuracy'] > 90).sum()} ({(df_results['accuracy'] > 90).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 80%:       {(df_results['accuracy'] > 80).sum()} ({(df_results['accuracy'] > 80).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 70%:       {(df_results['accuracy'] > 70).sum()} ({(df_results['accuracy'] > 70).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with CER < 10%:            {(df_results['cer'] < 10).sum()} ({(df_results['cer'] < 10).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with Vocabulary F1 > 80%:  {(df_results['vocab_f1'] > 80).sum()} ({(df_results['vocab_f1'] > 80).sum() / len(df_results) * 100:.1f}%)")
else:
    print("⚠ No data for analysis")


TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy  cer  wer  vocab_f1
sample_1727     100.0  0.0  0.0     100.0
sample_1384     100.0  0.0  0.0     100.0
sample_0888     100.0  0.0  0.0     100.0
sample_0020     100.0  0.0  0.0     100.0
sample_0137     100.0  0.0  0.0     100.0

TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy        cer        wer  vocab_f1
sample_2490 42.216359  55.188679 100.000000       0.0
sample_0924  0.000000 721.777003 820.512821       0.0
sample_0092  1.428005 465.668203 100.000000       0.0
sample_1517  3.516091 598.194131 829.508197       0.0
sample_2224  4.260985 194.651163 314.545455       0.0

BASIC STATISTICS
Samples with accuracy > 90%:       241 (43.1%)
Samples with accuracy > 80%:       287 (51.3%)
Samples with accuracy > 70%:       311 (55.6%)
Samples with CER < 10%:            356 (63.7%)
Samples with Vocabulary F1 > 80%:  275 (49.2%)


## 9. Example: Detailed View of a Sample

Run this cell to see detailed results for a specific sample:

In [10]:
# Change this to any sample_id you want to inspect
SAMPLE_TO_INSPECT = results_list[0]['sample_id'] if len(results_list) > 0 else None

if SAMPLE_TO_INSPECT:
    sample_result = next((r for r in results_list if r['sample_id'] == SAMPLE_TO_INSPECT), None)

    if sample_result:
        print("=" * 80)
        print(f"DETAILED RESULTS FOR: {SAMPLE_TO_INSPECT}")
        print("=" * 80)

        print(f"\n📄 Expected Text ({sample_result['expected_length']} chars):")
        print("-" * 80)
        print(sample_result['expected_text'])

        print(f"\n🤖 Predicted Text ({sample_result['predicted_length']} chars):")
        print("-" * 80)
        print(sample_result['predicted_text'])

        print(f"\n{format_metrics_display(sample_result)}")
else:
    print("⚠ No samples available for inspection")

DETAILED RESULTS FOR: sample_0246

📄 Expected Text (1069 chars):
--------------------------------------------------------------------------------
Patient: Albert Schuster Datum: 29.10.2025 29.10.2025, Hr. Schuster, 92 J., Zi. 32, PG 5
KOMPLETTE IMMOBILITÄT, bettlägerig seit Schlaganfall 2024. Grundpflege 7:00 Uhr: -
Ganzwaschwäsche im Bett, Pat. kann nicht mithelfen - Inkontinenzversorgung (Windel, Bettschutz)
- Katheter suprapubisch, Urin klar-gelb, ca. 800ml/24h - Mundpflege intensiv Lagerungsplan:
7:00 rechts - 10:00 links - 13:00 rechts - 16:00 links - 19:00 rechts Anti-Dekubitus-Matratze
im Einsatz. DEKUBITUS-STATUS: - Sakral Grad 2, ca. 3x4 cm, Wundversorgung mit HydroClean
- Beide Fersen freigelagert mit Gelkissen - Haut ansonsten intakt Ernährung: - Vollkost püriert
über Löffel - Frühstück: 5 Löffel Brei - Mittag: ca. 10 Löffel Gemüsebrei, 3 Löffel Kartoffelpüree -
Trinkmenge: 600ml (in kleinen Schlucken) Kommunikation: Pat. reagiert mit Augenkontakt, kann
nicht sprechen. Verst

---
## ℹ️ Usage Instructions

### Quick Start (Step by Step)
1. **Cell 1**: Install dependencies (uncomment, run once, then recomment)
2. **Cell 3**: Run imports and verify paths
3. **Cell 5**: Configure paths and settings  
   _(Optional: set `USE_TEST_SET = False` and `LIMIT_SAMPLES = 10` for a quick smoke-test)_
4. **Cell 7**: Load model — **downloads ~2 GB on first run**, loads into GPU memory (~3 GB VRAM for bfloat16)
5. **Cell 9**: Load and validate image-label pairs
6. **Cell 11**: **Main processing** — runs OCR on all images (may take a while)
7. **Cell 13**: Calculate summary statistics
8. **Cell 15**: Save all results
9. **Cells 17 & 19**: View quick analysis and inspect an individual sample

### ⚠️ transformers Version Note
LightOnOCR-2 requires **transformers ≥ 5.0**.  
Surya OCR requires **transformers < 5.0**.  
These cannot coexist in the same venv — restart the kernel before switching between the two notebooks.

### Expected Output Files
- `results/metrics/lightonocr/summary.csv` — All samples with metrics
- `results/metrics/lightonocr/summary.json` — Aggregate statistics
- `results/metrics/lightonocr/detailed_results.json` — Full OCR output
- `results/metrics/lightonocr/individual/` — Per-sample JSON files

### Model Info
- **Model**: `lightonai/LightOnOCR-2-1B`
- **Architecture**: End-to-end vision-language model (Mistral-3 base)
- **VRAM required**: ~3 GB (bfloat16 on CUDA)
- **Speed**: ~5.7 pages/s on H100 (single-page documents)